# 28 oct

downloaded the disease-target and the drug target sections now I need to parse and extract only approved targets with neuro indications

This can form the target disease junction table 

In [15]:
# for target disease file
import pandas as pd
import re

def add_approved_neuro(diseases):
    icd_term = r"\[ICD-11:\s*([68][A-Z0-9]*(?:\.[A-Z0-9]+)?(?:\s*-\s*[A-Z0-9]+(?:\.[A-Z0-9]+)?)?)\s*\]"
    icds = []
    for disease in diseases:
        if disease.startswith('Approved'):
            if re.search(icd_term, disease):
                pat = re.compile(icd_term)
                icd = pat.findall(disease)
                icds.append(icd)
    return icds

with open('/home/school/masters/Scripts/Theraputic_Target_Database/P1-06-Target_disease.txt') as f:
    target_disease_df = pd.DataFrame(columns=['ttd_target_id','field_name','field'])
    for line_num, line in enumerate(f, start=1):
        if line_num < 23:
            continue
        if line.strip():
            line = line.strip()
            line = line.split(maxsplit=2)
            target_disease_df.loc[len(target_disease_df)] = line 

target_disease_df = target_disease_df.loc[~target_disease_df['field_name'].eq('TARGNAME')].copy()
target_disease_df = target_disease_df.loc[~target_disease_df['field_name'].eq('TARGETID')].copy()

target_disease_df =target_disease_df.drop(columns=['field_name'])

target_disease_df.groupby('ttd_target_id').agg(add_approved_neuro)
target_disease_df = target_disease_df.loc[target_disease_df['field'].notna()].copy()


now need to sort the targets and ectract 

In [26]:
import pandas as pd

with open('/home/school/masters/Scripts/Theraputic_Target_Database/P1-01-TTD_target_download.txt') as f:
    target_df = pd.DataFrame(columns=(['ttd_id','field_name','field']))
    for line_num, line in enumerate(f, start=1):
        if line_num > 33:
            if line.strip():
                line = line.strip()
                line = line.split(maxsplit=2)
                target_df.loc[len(target_df)] = line

target_df = target_df.loc[~target_df['field_name'].isin(['TARGETID','FORMERID','DRUGINFO','SEQUENCE'])].copy()

target_df = target_df.groupby(['ttd_id','field_name'])['field'].agg(list).unstack('field_name').reset_index()
target_df

field_name,ttd_id,BIOCLASS,ECNUMBER,FUNCTION,GENENAME,PDBSTRUC,SYNONYMS,TARGNAME,TARGTYPE,UNIPROID
0,T00032,NaN,NaN,[Appears to form an integral part of the miner...,[SPP1],[3DSF; 3CXD],[Uropontin; Urinary stone protein; Secreted ph...,[Osteopontin (SPP1)],[Literature-reported],[OSTP_HUMAN]
1,T00033,[Growth factor],NaN,[TGF alpha is a mitogenic polypeptide that is ...,[TGFA],[5KN5; 4TGF; 3TGF; 3.00E+50; 2TGF],[TGF-alpha (40-89); TGF type 1 (40-89); Protra...,[Transforming growth factor alpha (TGFA)],[Clinical trial],[TGFA_HUMAN]
2,T00037,[Methyltransferase],[EC 2.1.1.-],[Catalyzes the methyl transfer from S-adenosyl...,[Fung erg6],NaN,[erg6; SAM:SMT; S-adenosyl-L-methionine:sterol...,[Fungal Sterol 24-C-methyltransferase (Fung er...,[Literature-reported],[ERG6_PNEC8]
3,T00039,[mRNA target],NaN,[Promotes proliferation and differentiation of...,[CTGF],NaN,[Long (mRNA); Insulin-like growth factor-bindi...,[CTGF messenger RNA (CTGF mRNA)],[Clinical trial],[CTGF_HUMAN]
4,T00064,[Non-coding RNA target],NaN,NaN,NaN,NaN,NaN,[microRNA hsa-miR-199a (MIR199a)],[Literature-reported],NaN
...,...,...,...,...,...,...,...,...,...,...
4293,T99948,[Immunoglobulin],NaN,[As a ligand for the inhibitory receptor PDCD1...,[CD274],[6R3K; 6NP9; 6NOS; 6NOJ; 6NNV],[hPD-L1; Programmed death ligand 1; PDL1; PDCD...,[Programmed cell death 1 ligand 1 (PD-L1)],[Successful],[PD1L1_HUMAN]
4294,T99954,[GPCR rhodopsin],NaN,[The activity of this receptor is mediated by ...,[PTGIR],NaN,[Prostanoid IP receptor; Prostaglandin I2 rece...,[Prostacyclin receptor (PTGIR)],[Successful],[PI2R_HUMAN]
4295,T99976,NaN,NaN,NaN,NaN,NaN,NaN,[Human immunodeficiency virus Cell fusion (HIV...,[Literature-reported],NaN
4296,T99989,[Kinase],[EC 2.7.11.1],[Can phosphorylate a large number of proteins....,[CSNK1E],[4HOK; 4HNI],[Casein kinase I isoform epsilon; CKIe; CKI-ep...,[Casein kinase I epsilon (CSNK1E)],[Successful],[KC1E_HUMAN]


now handling the drug-diseases

In [ ]:
import pandas as pd
import re

def get_icd(value):
    if pd.isna(value):
        return pd.NA
    icd_term = r"\[ICD-11:\s*([68][A-Z0-9]*(?:\.[A-Z0-9]+)?(?:\s*-\s*[A-Z0-9]+(?:\.[A-Z0-9]+)?)?)\s*\]"
    icds = []
    if not re.search(icd_term, value):
        return pd.NA
    values = value.split('|')
    for value in values:
        phase = ''
        codes = []
        if re.search(icd_term, value):
            pat = re.compile(icd_term)
            codes = pat.findall(value)
            if 'Approved in' in value:
                vals = value.split()
                phase = ' '.join(vals[-3:])
            elif 'Phase' in value:
                vals = value.split()
                phase = ' '.join(vals[-2:])
            elif 'Approved' in value:
                vals = value.split()
                phase = ' '.join(vals[-1:])
            for code in codes:
                icds.append(f'{code},{phase}')
    return icds
        

with open('/home/school/masters/Scripts/Theraputic_Target_Database/P1-05-Drug_disease.txt') as f:
    drug_disease_df = pd.DataFrame(columns = ['ttd_id','name','indication'])
    use = []
    d_id = None
    d_name = None
    for line_num, line in enumerate(f,start=1):
        if line_num > 22:
            line = line.strip()
            if not line: 
                continue
            line = line.split(maxsplit=1) if line else line
            
            if line[0] == 'TTDDRUID':
                if use and d_id is not None:
                    entry = [d_id,d_name,'|'.join(use)]
                    drug_disease_df.loc[len(drug_disease_df)] = entry
                d_id = line[1]
                d_name = None
                use = []
            elif line[0] == 'DRUGNAME':
                d_name = line[1]
            elif line[0] == 'INDICATI':
                use.append(line[1])
    if use and d_id is not None:
        entry = [d_id, d_name, '|'.join(use)]
        drug_disease_df.loc[len(drug_disease_df)] = entry

drug_disease_df['icds'] = drug_disease_df.apply(lambda row: get_icd(row.indication),axis = 1)
drug_disease_df = drug_disease_df.explode('icds', ignore_index=True).dropna(subset=['icds'])
if drug_disease_df.empty:
    drug_disease_df['icd'] = pd.Series(dtype='string')
    drug_disease_df['phase'] = pd.Series(dtype='string')
else:
    drug_disease_df[['icd', 'phase']] = drug_disease_df['icds'].str.split(',', n=1, expand=True)


drug_disease_df = drug_disease_df.drop(columns=['icds','indication'])

Now need to extract the drugs

In [8]:
import pandas as pd

with open('/home/school/masters/Scripts/Theraputic_Target_Database/P1-02-TTD_drug_download.txt') as f:
    drugs_df = pd.DataFrame(columns=['ttd_id','field_name','field'])
    for line_num, line in enumerate(f,start=1):
        if line_num > 28:
            if line.strip():
                line = line.split(maxsplit=2)
                ttid = line[0]
                field_name = line[1]
                field = line[2]
                drugs_df.loc[len(drugs_df)] = [ttid,field_name,field]

drugs_df = drugs_df.loc[~drugs_df['field_name'].isin(['TRADNAME','DRUGCOMP', 'DRUGTYPE'])].copy()

drugs_df = drugs_df.groupby(['ttd_id','field_name'])['field'].agg(list).unstack('field_name').reset_index()
drugs_df


KeyboardInterrupt: 